# 🛠️ Notebook: Introduction to Function Calling

In this notebook we learn how to make LLMs call functions.

## 📚 Sources

- [OpenAI: Function Calling API](https://platform.openai.com/docs/guides/function-calling)
- [Ollama: Function Calling](https://ollama.com/blog/functions-as-tools)

---

Good luck trying out the Function Calling features! 🤗

Let's start by installing the Python package from ollama. Unlike the previous notebooks where we used the OpenAI API, we'll use the Ollama API for a change in this notebook.


Just like `ollama`, the `openai` library also provides the ability to send requests to a server with an LLM API. Note: `openai` supports Function Calling just like `ollama`.

[https://platform.openai.com/docs/guides/function-calling](https://platform.openai.com/docs/guides/function-calling)

In [1]:
%%capture
!pip install ollama

In [ ]:
LLM_URL = "http://localhost:11434"
# Not every LLM supports Function Calling. 
# Gemma3, the LLM from the last exercise, for example, was not trained for this. The open-source LLM "gpt-oss:20b" from OpenAI, on the other hand, was.
LLM_MODEL = "gpt-oss:20b"

In [3]:
from ollama import Client

client = Client(
  host=LLM_URL
)

### 1. Example: Tool Call with Stock Prices

Let's first define a list of tools. In our example, only **one** tool that returns the current stock price for a given symbol. 

The `openai` library expects each tool in a specific format. We define the tool as a dictionary with the following keys:

* `type`: OpenAI also offers other tool types in principle (e.g. `web_search`), but we only use `function` here.
* `name`: The name of the tool.
* `parameters`: The parameters that the function expects.
* `required`: Which parameters must be provided.

In [4]:
tools = [{
    'type': 'function',
    'function': {
            'name': 'get_stock_price', # Name of the tool
            'description': 'Get the current stock price for a company', # Description of the tool
            'parameters': {
                'type': 'object',
                'properties': {
                    'symbol': { # The function expects exactly one parameter, "symbol". Symbol is the stock ticker, e.g. "AAPL" for Apple.
                        'type': 'string', # Type of the parameter
                    },
                },
                'required': ['symbol'], # Required parameters
            },
    },
}]

In [5]:
# Let's start with an example where we query the current SAP stock price.
response = client.chat(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': 'What is the current SAP stock price?'}], # User query
    tools=tools, # We pass the tools that the model can use
)

# We can now see if and how the model called the tool.
tool_calls = response['message']['tool_calls']
print(f"Tool calls: {tool_calls}")

# Output arguments of the tool call and function
print(f"Function name: {tool_calls[0].function.name}")
print(f"Function arguments: {tool_calls[0].function.arguments}")

Tool calls: [ToolCall(function=Function(name='get_stock_price', arguments={'symbol': 'SAP'}))]
Function name: get_stock_price
Function arguments: {'symbol': 'SAP'}


In [6]:
# We can try asking an irrelevant question that has nothing to do with stocks.
# In that case, tool_calls is None because the LLM doesn't need to call a tool.
response = client.chat(
    model=LLM_MODEL,
    messages=[{'role': 'user', 'content': 'What is 1+1?'}],
    tools=tools,
)

# Let's look at the output. In the output we see that tool_calls is None.
print(response)

model='gpt-oss:20b' created_at='2025-12-22T18:47:06.48053Z' done=True done_reason='stop' total_duration=477149375 load_duration=138219792 prompt_eval_count=131 prompt_eval_duration=37479666 eval_count=30 eval_duration=292984417 message=Message(role='assistant', content='1\u202f+\u202f1\u202f=\u202f2.', thinking='User asks a simple math question. Provide answer.', images=None, tool_name=None, tool_calls=None)


## Exercise: Tool Execution - Let's execute the function

So far, we have only implemented that the LLM possibly calls a tool with parameters, but the actual function behind it is not executed.

Let's now implement a small pipeline that actually executes a function when the LLM calls a tool. We'll now implement a tool that retrieves information about countries.

Proceed with the implementation as follows:

1. Define a tool `get_country_info` in the JSON schema with parameter `country`
2. Implement the function `get_country_info_from_api` that retrieves country info from the API
3. Build a simple workflow that queries the LLM and then calls the function

This is how the API works:

In [7]:
import requests

country = "Germany"  # Also possible: France, Italy, Japan, ...
url = f"https://restcountries.com/v3.1/name/{country}"
response = requests.get(url)
data = response.json()

# API returns a list, we take the first element
country_data = data[0]
print(f"Name: {country_data['name']['common']}")
print(f"Capital: {country_data['capital'][0]}")
print(f"Population: {country_data['population']}")
print(f"Region: {country_data['region']}")

# Write your code here ...

Name: Germany
Capital: Berlin
Population: 83491249
Region: Europe


<details>
<summary><b>Show solution</b></summary>

```python
# 1. Tool Definition
tools = [{
    'type': 'function',
    'function': {
        'name': 'get_country_info',
        'description': 'Retrieves information about a country',
        'parameters': {
            'type': 'object',
            'properties': {
                'country': {
                    'type': 'string',
                    'description': 'The name of the country (e.g. Germany, France, Japan)'
                }
            },
            'required': ['country']
        }
    }
}]

# 2. API Function
def get_country_info_from_api(country):
    url = f"https://restcountries.com/v3.1/name/{country}"
    response = requests.get(url)
    data = response.json()[0]

    # We can summarize the information into a text:
    info = f"Country: {data['name']['common']}, Capital: {data['capital'][0]}, Population: {data['population']}, Region: {data['region']}"
    return info


# 3. Workflow
messages = [{'role': 'user', 'content': 'Tell me something about Norway.'}]

# Query LLM
response = client.chat(model=LLM_MODEL,
                       messages=messages,
                       tools=tools,
                       options={"temperature": 0.0})

# Was a tool called? Let's check if the key ['message']['tool_calls'] exists
if response['message']['tool_calls']:
    country = response['message']['tool_calls'][0]['function']['arguments']['country']

    # Execute function
    country_info = get_country_info_from_api(country)

    # Create prompt for final answer
    prompt = f"Here is the information about {country}:\n{country_info}.\nPlease summarize this information briefly in two sentences."

    # Back to the LLM
    messages = [{'role': 'user', 'content': prompt}]
    final_response = client.chat(
        model=LLM_MODEL, messages=messages, options={"temperature": 0.0})

    print(f"\nFinal answer: {final_response['message']['content']}")
```

</details>